# Diffusion-Regularized PINNs Demo (2D)

This notebook demonstrates the comparison between:
- **Poisson**: Poisson-only inverse PINN (P-iPINN in paper)
- **Coupled**: Poisson + Diffusion-Decay regularized PINN (PD-iPINN in paper)

**Domain**: 2D spatial domain (x, y) + time

**Runtime**: ~15 minutes on Colab GPU

In [ ]:
# =============================================================================
# IMPORTANT: Set backend BEFORE importing deepxde
# =============================================================================
import os
os.environ['DDE_BACKEND'] = 'tensorflow.compat.v1'

# Install dependencies (Colab)
!pip install deepxde==1.14.0 --quiet

In [ ]:
# =============================================================================
# Imports
# =============================================================================
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print(f"DeepXDE: {dde.__version__}")
print(f"Backend: {dde.backend.backend_name}")

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

# Random seed for reproducibility
SEED = 42

# Noise level (0.0 to 1.0)
NOISE_LEVEL = 0.5  # 50% noise

# Grid resolution (2D spatial + time)
NUM_X = 21   # Spatial grid points (x)
NUM_Y = 21   # Spatial grid points (y)
NUM_T = 51   # Temporal grid points

# Training settings
EPOCHS = 5000
DISPLAY_EVERY = 1000
NUM_DOMAIN = 5000  # More collocation points for 2D

# Physical parameters
EPS = 1.0        # Permittivity
D_COEFF = 0.01   # Diffusion coefficient [m²/s]
K_COEFF = 0.5    # Decay coefficient [1/s]

# Derived parameters (2D: λ = D(π² + π²) + k = 2Dπ² + k)
DECAY_RATE = D_COEFF * 2 * (np.pi ** 2) + K_COEFF

# Domain bounds
X_MIN, X_MAX = -1.0, 1.0
Y_MIN, Y_MAX = -1.0, 1.0
T_MIN, T_MAX = 0.0, 1.0

print(f"Configuration: noise={NOISE_LEVEL*100:.0f}%, grid=({NUM_X}×{NUM_Y}×{NUM_T})")
print(f"Total points: {NUM_X * NUM_Y * NUM_T}")
print(f"Decay rate λ = 2Dπ² + k = {DECAY_RATE:.4f}")

In [ ]:
# =============================================================================
# Exact Solutions
# =============================================================================

def phi_ex_func(X):
    """Exact potential: φ_ex(x,y,t) = sin(πx) sin(πy) exp(-λt)"""
    x, y, t = X[:, 0:1], X[:, 1:2], X[:, 2:3]
    return np.sin(np.pi * x) * np.sin(np.pi * y) * np.exp(-DECAY_RATE * t)

def rho_ex_func(X):
    """Exact charge density: ρ_ex(x,y,t) = 2επ² sin(πx) sin(πy) exp(-λt)"""
    x, y, t = X[:, 0:1], X[:, 1:2], X[:, 2:3]
    return EPS * 2 * (np.pi ** 2) * np.sin(np.pi * x) * np.sin(np.pi * y) * np.exp(-DECAY_RATE * t)

def add_noise(phi_clean, noise_level, seed):
    """Add relative Gaussian noise: φ_obs = φ_ex + σ·max|φ_ex|·η"""
    np.random.seed(seed)
    if noise_level == 0:
        return phi_clean.copy()
    phi_scale = np.max(np.abs(phi_clean))
    noise = noise_level * phi_scale * np.random.randn(*phi_clean.shape)
    return phi_clean + noise

In [ ]:
# =============================================================================
# PDE Residuals (2D)
# =============================================================================

def pde_poisson_only(X, y):
    """2D Poisson model: -(∂²φ/∂x² + ∂²φ/∂y²) - ρ/ε = 0"""
    phi_xx = dde.grad.hessian(y, X, component=0, i=0, j=0)
    phi_yy = dde.grad.hessian(y, X, component=0, i=1, j=1)
    rho = y[:, 1:2]
    return -phi_xx - phi_yy - rho / EPS

def pde_coupled(X, y):
    """2D Coupled model: Poisson + Diffusion-Decay

    1. Poisson: -(∂²φ/∂x² + ∂²φ/∂y²) - ρ/ε = 0
    2. Diffusion-Decay: ∂ρ/∂t - D(∂²ρ/∂x² + ∂²ρ/∂y²) + kρ = 0
    """
    # Poisson equation
    phi_xx = dde.grad.hessian(y, X, component=0, i=0, j=0)
    phi_yy = dde.grad.hessian(y, X, component=0, i=1, j=1)
    rho = y[:, 1:2]

    # Diffusion-Decay equation
    rho_t = dde.grad.jacobian(y, X, i=1, j=2)  # ∂ρ/∂t (t is index 2)
    rho_xx = dde.grad.hessian(y, X, component=1, i=0, j=0)
    rho_yy = dde.grad.hessian(y, X, component=1, i=1, j=1)

    res_poisson = -phi_xx - phi_yy - rho / EPS
    res_diffusion = rho_t - D_COEFF * (rho_xx + rho_yy) + K_COEFF * rho

    return [res_poisson, res_diffusion]

In [ ]:
# =============================================================================
# Training Function (2D)
# =============================================================================

def train_model(model_type, phi_obs, XYT_data, seed):
    tf.compat.v1.reset_default_graph()
    dde.config.set_random_seed(seed)
    np.random.seed(seed)

    # 2D Geometry
    geom = dde.geometry.Rectangle([X_MIN, Y_MIN], [X_MAX, Y_MAX])
    timedomain = dde.geometry.TimeDomain(T_MIN, T_MAX)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)

    # Observation constraint
    observe_phi = dde.icbc.PointSetBC(XYT_data, phi_obs, component=0)

    # Model-specific settings
    if model_type == 'Poisson':
        pde_fn = pde_poisson_only
        loss_weights = [1, 1000]
    else:  # Coupled
        pde_fn = pde_coupled
        loss_weights = [1, 1, 1000]

    # Data
    data = dde.data.TimePDE(
        geomtime, pde_fn, [observe_phi],
        num_domain=NUM_DOMAIN, num_boundary=0, num_initial=0,
        anchors=XYT_data, num_test=5000
    )

    # Network: input is 3 (x, y, t), output is 2 (φ, ρ)
    net = dde.nn.PFNN(
        [3, [64, 64], [64, 64], [64, 64], [64, 64], 2],
        "tanh", "Glorot uniform"
    )

    # Compile and train
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3, loss_weights=loss_weights)

    print(f"\nTraining {model_type} (2D)...")
    losshistory, _ = model.train(epochs=EPOCHS, display_every=DISPLAY_EVERY)

    # Predict
    output = model.predict(XYT_data)
    phi_pred = output[:, 0:1]
    rho_pred = output[:, 1:2]

    tf.keras.backend.clear_session()

    return phi_pred, rho_pred, losshistory

In [ ]:
# =============================================================================
# Generate Data (2D)
# =============================================================================

# Create 3D grid (x, y, t)
x_grid = np.linspace(X_MIN, X_MAX, NUM_X)
y_grid = np.linspace(Y_MIN, Y_MAX, NUM_Y)
t_grid = np.linspace(T_MIN, T_MAX, NUM_T)

X_mesh, Y_mesh, T_mesh = np.meshgrid(x_grid, y_grid, t_grid, indexing='ij')
XYT_data = np.hstack([
    X_mesh.flatten()[:, None],
    Y_mesh.flatten()[:, None],
    T_mesh.flatten()[:, None]
])

# Ground truth
phi_ex = phi_ex_func(XYT_data)
rho_ex = rho_ex_func(XYT_data)

# Add noise
phi_obs = add_noise(phi_ex, NOISE_LEVEL, SEED)

In [ ]:
# =============================================================================
# Train Both Models
# =============================================================================

# Train Poisson-only model
phi_pred_P, rho_pred_P, loss_P = train_model('Poisson', phi_obs, XYT_data, SEED)

# Train Coupled model (Poisson + Diffusion-Decay)
phi_pred_C, rho_pred_C, loss_C = train_model('Coupled', phi_obs, XYT_data, SEED)

In [ ]:
# =============================================================================
# Compute Metrics
# =============================================================================

def relative_L2_error(pred, exact):
    """Relative L2 error: ||pred - exact||₂ / ||exact||₂"""
    return np.sqrt(np.mean((pred - exact)**2)) / np.sqrt(np.mean(exact**2))

# Poisson model metrics
phi_L2_P = relative_L2_error(phi_pred_P, phi_ex)
rho_L2_P = relative_L2_error(rho_pred_P, rho_ex)

# Coupled model metrics
phi_L2_C = relative_L2_error(phi_pred_C, phi_ex)
rho_L2_C = relative_L2_error(rho_pred_C, rho_ex)

print("\n" + "="*50)
print(f"Results (2D, noise = {NOISE_LEVEL*100:.0f}%)")
print("="*50)
print(f"{'Model':<12} {'φ L2 Error':<15} {'ρ L2 Error':<15}")
print("-"*50)
print(f"{'Poisson':<12} {phi_L2_P*100:>10.2f}%     {rho_L2_P*100:>10.2f}%")
print(f"{'Coupled':<12} {phi_L2_C*100:>10.2f}%     {rho_L2_C*100:>10.2f}%")
print("="*50)
print(f"\nρ error reduction: {(1 - rho_L2_C/rho_L2_P)*100:.1f}%")

In [ ]:
# Setup for plotting (2D)
def reshape_to_grid_2d(arr):
    return arr.reshape(NUM_X, NUM_Y, NUM_T)

phi_ex_grid = reshape_to_grid_2d(phi_ex)
rho_ex_grid = reshape_to_grid_2d(rho_ex)
phi_obs_grid = reshape_to_grid_2d(phi_obs)
rho_pred_P_grid = reshape_to_grid_2d(rho_pred_P)
rho_pred_C_grid = reshape_to_grid_2d(rho_pred_C)

import matplotlib as mpl
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['STIXGeneral'],
    'mathtext.fontset': 'stix',
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'axes.linewidth': 0.8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.fontsize': 7,
    'legend.frameon': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

DOUBLE_COL = 6.69
C_TRUE, C_P, C_C = '#000000', '#D55E00', '#0072B2'

# Time indices for visualization
t_indices = [0, NUM_T//4, NUM_T//2, 3*NUM_T//4, NUM_T-1]
t_values = [t_grid[i] for i in t_indices]

print(f"Visualization time points: {t_values}")

In [ ]:
# =============================================================================
# Figure 1
# =============================================================================
fig, axes = plt.subplots(1, 4, figsize=(DOUBLE_COL, DOUBLE_COL * 0.28))

t_idx = 0  # t = 0
vmin_phi, vmax_phi = phi_ex_grid[:, :, t_idx].min(), phi_ex_grid[:, :, t_idx].max()
vmin_rho, vmax_rho = rho_ex_grid[:, :, t_idx].min(), rho_ex_grid[:, :, t_idx].max()

# (a) ρ_ex
ax = axes[0]
ax.imshow(rho_ex_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
          vmin=vmin_rho, vmax=vmax_rho, cmap='RdBu_r')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_title(r'(a) $\rho_{ex}$')
ax.set_aspect('equal')

# (b) φ_ex
ax = axes[1]
ax.imshow(phi_ex_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
          vmin=vmin_phi, vmax=vmax_phi, cmap='RdBu_r')
ax.set_xlabel(r'$x$')
ax.set_title(r'(b) $\phi_{ex}$')
ax.set_aspect('equal')

# (c) φ_obs
ax = axes[2]
ax.imshow(phi_obs_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
          vmin=vmin_phi, vmax=vmax_phi, cmap='RdBu_r')
ax.set_xlabel(r'$x$')
ax.set_title(rf'(c) $\phi_{{obs}}$ ($\sigma$={int(NOISE_LEVEL*100)}%)')
ax.set_aspect('equal')

# (d) L2 error bar chart
ax = axes[3]
L2_P_list, L2_C_list = [], []
for t_idx in t_indices:
    rho_ex_slice = rho_ex_grid[:, :, t_idx].flatten()
    L2_P = np.sqrt(np.mean((rho_pred_P_grid[:, :, t_idx].flatten() - rho_ex_slice)**2)) / np.sqrt(np.mean(rho_ex_slice**2))
    L2_C = np.sqrt(np.mean((rho_pred_C_grid[:, :, t_idx].flatten() - rho_ex_slice)**2)) / np.sqrt(np.mean(rho_ex_slice**2))
    L2_P_list.append(L2_P * 100)
    L2_C_list.append(L2_C * 100)

x_pos = np.arange(len(t_values))
width = 0.35
ax.bar(x_pos - width/2, L2_P_list, width, color=C_P, label='Poisson')
ax.bar(x_pos + width/2, L2_C_list, width, color=C_C, label='Coupled')
ax.set_xlabel(r'$t$')
ax.set_ylabel(r'Relative $L_2$ Error (%)')
ax.set_title('(d) Error comparison')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{t:.2f}' for t in t_values])
ax.set_ylim(0, max(L2_P_list) * 1.2)
ax.legend(loc='upper left', fontsize=5)

fig.tight_layout()
plt.savefig('demo_2d_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Figure 2
# =============================================================================
fig, axes = plt.subplots(3, 5, figsize=(DOUBLE_COL, DOUBLE_COL * 0.65))

for col, (t_idx, t_val) in enumerate(zip(t_indices, t_values)):
    vmin = rho_ex_grid[:, :, t_idx].min()
    vmax = rho_ex_grid[:, :, t_idx].max()

    # Row 0: True
    ax = axes[0, col]
    ax.imshow(rho_ex_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
              vmin=vmin, vmax=vmax, cmap='RdBu_r')
    ax.set_title(f'$t = {t_val:.2f}$')
    ax.set_aspect('equal')
    if col == 0:
        ax.set_ylabel('True\n$y$')
    ax.set_xticks([])
    ax.set_yticks([] if col > 0 else [-1, 0, 1])

    # Row 1: Poisson
    ax = axes[1, col]
    ax.imshow(rho_pred_P_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
              vmin=vmin, vmax=vmax, cmap='RdBu_r')
    ax.set_aspect('equal')
    if col == 0:
        ax.set_ylabel('Poisson\n$y$')
    ax.set_xticks([])
    ax.set_yticks([] if col > 0 else [-1, 0, 1])

    # Row 2: Coupled
    ax = axes[2, col]
    ax.imshow(rho_pred_C_grid[:, :, t_idx].T, origin='lower', extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
              vmin=vmin, vmax=vmax, cmap='RdBu_r')
    ax.set_aspect('equal')
    ax.set_xlabel(r'$x$')
    if col == 0:
        ax.set_ylabel('Coupled\n$y$')
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([] if col > 0 else [-1, 0, 1])

fig.suptitle(rf'$\rho$ reconstruction over time (noise $\sigma = {int(NOISE_LEVEL*100)}\%$)',
             fontsize=9, y=1.02)
fig.tight_layout()
plt.savefig('demo_2d_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

This 2D demo shows that:

1. **Poisson** model amplifies noise in 2D, producing spatial artifacts
2. **Coupled** model provides stable reconstruction across the 2D domain

The diffusion-decay equation includes the 2D Laplacian:

$$\frac{\partial \rho}{\partial t} = D \left( \frac{\partial^2 \rho}{\partial x^2} + \frac{\partial^2 \rho}{\partial y^2} \right) - k\rho$$

---

### 1D vs 2D Comparison

| Aspect | 1D | 2D |
|--------|----|----|
| Domain | x ∈ [-1, 1] | (x, y) ∈ [-1, 1]² |
| Input | (x, t) | (x, y, t) |
| Decay rate | λ = Dπ² + k | λ = 2Dπ² + k |
| Laplacian | ∂²/∂x² | ∂²/∂x² + ∂²/∂y² |

---

### Model Naming Convention

| Code | Paper | Description |
|------|-------|-------------|
| `Poisson` | P-iPINN | Poisson-only inverse PINN |
| `Coupled` | PD-iPINN | Poisson + Diffusion-Decay |